# STAT5106 Final Project — Part C: Exploratory Data Analysis & Visualization

**Research question:** How do review ratings and five-star rates (`five_star`) vary across price segments (`price_segment`), product categories (`category`), and reviewer skin types (`skin_type_cat`) in Sephora skincare reviews?

**Input:** `analysis_ready.parquet` from Part B (1,093,847 reviews × 52 columns).

**Outputs:** Summary tables in `outputs/` and four figures in `outputs/figures/`, plus key findings for the final report. Note: `rating`, `recommended`, and 2023 snapshot flags (`new`, `online_only`, `limited_edition`, `out_of_stock`, `sephora_exclusive`) are not used as predictors, and product-level counts (`product_id`) are tracked alongside review counts because train/test splits, clustering, and bootstrap resampling are product-based.

In [ ]:
# Install dependencies once with: python -m pip install -r requirements.txt


## Step 1. Load Part B's analysis-ready dataset and check outcomes

Load `analysis_ready.parquet` using PyArrow/pandas so ordered categories and reference levels from Part B are preserved. Confirm that all 1,093,847 rows and the main outcome `five_star` (`rating == 5`) are present without unexpected missing values.

In [ ]:
# Find only this project (never search the user's home directory).
import sys
from pathlib import Path
_PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_paths.py").is_file()), None)
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook within the project folder.")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from project_paths import ROOT, RESULTS_DIR, ANALYSIS_PATH, RAW_DIR, PROCESSED_DIR, require_analysis, require_merged, output_dir

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

data_path = require_analysis()
print("Using data path:", data_path)
OUT_DIR = output_dir("C")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 2. Data Reading and Basic Validation
table = pq.read_table(data_path)
df = table.to_pandas(types_mapper=None)

assert len(df) == 1_093_847, f"Expected 1,093,847 rows, got {len(df)}"
assert df["rating"].isna().sum() == 0, "Unexpected missing values in rating"
assert df["five_star"].notna().all()
assert (df["five_star"] == (df["rating"] == 5).astype(int)).all()


## Step 2. Overall rating distribution (1–5 stars)

Examine the distribution of star ratings (`1` to `5`) across all reviews to establish the baseline rating level, dispersion, and skewness (whether ratings are concentrated at the top end). Export the star-level count and share table and plot `fig1_rating_distribution.png`.

In [ ]:
# Statistical Table of 1–5 Star Distribution
rating_counts = df["rating"].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
rating_shares = rating_counts / len(df) * 100

dist_df = pd.DataFrame({
    "star": rating_counts.index,
    "n_reviews": rating_counts.values,
    "share_pct": rating_shares.round(2).values,
    "cum_share_pct": rating_shares.cumsum().round(2).values,
})
dist_df["is_five_star"] = dist_df["star"] == 5

print("=== Overall Rating Distribution (1-5 Stars) ===")
print(dist_df.to_string(index=False))
print(f"\nMean rating: {df['rating'].mean():.3f}")
print(f"Median rating: {df['rating'].median():.1f}")
print(f"Std rating: {df['rating'].std():.3f}")
print(f"Five-star rate (five_star == 1): {df['five_star'].mean() * 100:.2f}%")

# Export statistical table

# Visualization: Overall Score Distribution Plot
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

colors = ["#D95F02", "#E69F00", "#F0E442", "#56B4E9", "#2B8CBE"]
bars = ax.bar(
    [f"{s}★" for s in dist_df["star"]],
    dist_df["share_pct"],
    color=colors,
    edgecolor="black",
    linewidth=0.7,
)

for bar, cnt, pct in zip(bars, dist_df["n_reviews"], dist_df["share_pct"]):
  ax.text(
      bar.get_x() + bar.get_width() / 2,
      bar.get_height() + 0.8,
      f"{pct:.1f}%\n(n={cnt:,})",
      ha="center",
      va="bottom",
      fontsize=9.5,
  )
    
ax.set_xticks(range(len(dist_df)))
ax.set_xticklabels([f"{s} Star" for s in dist_df["star"]])
ax.set_ylim(0, dist_df["share_pct"].max() * 1.18)
ax.set_xlabel("Star Rating")
ax.set_ylabel("Share of Reviews (%)")
ax.set_title(
    f"Rating Distribution (N = {len(df):,}, Five-Star Rate ="
    f" {df['five_star'].mean()*100:.2f}%)"
)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_rating_distribution.png")
plt.close()

print(f"\nSaved table & figure to: {OUT_DIR.resolve()}")

## Step 3. Five-star rate across price segments, categories, and skin types

Compare `five_star` rates against the overall baseline (~63.9%) across `price_segment` (4 fixed price bands), `category` (9 merged product categories), and `skin_type_cat` (5 skin-type groups including `'Not reported'`). Report both review counts (`n_reviews`) and unique product counts (`n_products`) for each group, and plot `fig2_price_and_skintype_five_star.png`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

overall_fsr = df["five_star"].mean() * 100


def summarize_five_star(data: pd.DataFrame, col: str, sort_desc: bool = True) -> pd.DataFrame:
  tbl = data.groupby(col, observed=False).agg(
      n_reviews=("five_star", "count"),
      n_products=("product_id", "nunique"),
      review_share_pct=("five_star", lambda s: len(s) / len(data) * 100),
      five_star_pct=("five_star", lambda s: s.mean() * 100),
  )
  tbl["diff_vs_overall_pp"] = tbl["five_star_pct"] - overall_fsr
  if sort_desc:
    tbl = tbl.sort_values("five_star_pct", ascending=False)
  return tbl.round({
      "review_share_pct": 2,
      "five_star_pct": 2,
      "diff_vs_overall_pp": 2,
  })


# Summary Table by Dimension
tab_price = summarize_five_star(df, "price_segment", sort_desc=False)
tab_cat = summarize_five_star(df, "category", sort_desc=True)
tab_skin = summarize_five_star(df, "skin_type_cat", sort_desc=True)

print("=== 1. Five-Star Rate by Price Segment ===")
print(tab_price.to_string())

print("\n=== 2. Five-Star Rate by Product Category ===")
print(tab_cat.to_string())

print("\n=== 3. Five-Star Rate by Skin Type ===")
print(tab_skin.to_string())


# Visualization: Comparison of 5-star rating percentages by Price Segment and Skin Type
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

price_order = [
    c
    for c in [
        "Budget (<$25)",
        "Mid ($25-49)",
        "Premium ($50-99)",
        "Luxury ($100+)",
    ]
    if c in tab_price.index
]
p_plot = tab_price.loc[price_order]

b0 = axes[0].bar(
    p_plot.index.astype(str), p_plot["five_star_pct"], color="#4C72B0"
)
axes[0].axhline(
    overall_fsr,
    color="#C44E52",
    ls="--",
    lw=1.5,
    label=f"Overall ({overall_fsr:.1f}%)",
)
for bar, (_, row) in zip(b0, p_plot.iterrows()):
  axes[0].text(
      bar.get_x() + bar.get_width() / 2,
      bar.get_height() + 0.5,
      f"{row['five_star_pct']:.1f}%\n(n={int(row['n_reviews']):,})",
      ha="center",
      va="bottom",
      fontsize=8.5,
  )
axes[0].set_ylim(0, max(p_plot["five_star_pct"].max(), overall_fsr) * 1.2)
axes[0].set_ylabel("Five-Star Rate (%)")
axes[0].set_title("Five-Star Rate by Price Segment")
axes[0].tick_params(axis="x", rotation=15)
axes[0].legend(loc="lower right", fontsize=9)

s_plot = tab_skin.sort_values("five_star_pct", ascending=True)
axes[1].barh(
    s_plot.index.astype(str), s_plot["five_star_pct"], color="#55A868"
)
axes[1].axvline(
    overall_fsr,
    color="#C44E52",
    ls="--",
    lw=1.5,
    label=f"Overall ({overall_fsr:.1f}%)",
)
for idx, (_, row) in enumerate(s_plot.iterrows()):
  axes[1].text(
      row["five_star_pct"] + 0.4,
      idx,
      f"{row['five_star_pct']:.1f}% (n={int(row['n_reviews']):,})",
      va="center",
      fontsize=8.5,
  )
axes[1].set_xlim(0, max(s_plot["five_star_pct"].max(), overall_fsr) * 1.3)
axes[1].set_xlabel("Five-Star Rate (%)")
axes[1].set_title("Five-Star Rate by Skin Type")
axes[1].legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_price_and_skintype_five_star.png")
plt.show()

## Step 4. Category ranking and cross-dimension interaction heatmap

Rank all nine product categories by five-star rate (`fig3_category_five_star.png`) with review and product counts labeled, and inspect two-way interaction cells (`price_segment` × `skin_type_cat`) for both five-star rate and unique product coverage (`fig4_price_skin_heatmap.png`). Because overall predictor discrimination is modest (AUC around 0.52), highlight descriptive group differences and product sample sizes rather than overstating predictive power.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

# Ranking of Product Categories by 5-Star Rating Percentage
c_plot = tab_cat.sort_values("five_star_pct", ascending=True)
fig, ax = plt.subplots(figsize=(10.5, 6), dpi=150)

ax.barh(c_plot.index.astype(str), c_plot["five_star_pct"], color="#8172B2")
ax.axvline(
    overall_fsr,
    color="#C44E52",
    ls="--",
    lw=1.5,
    label=f"Overall ({overall_fsr:.1f}%)",
)

for idx, (_, row) in enumerate(c_plot.iterrows()):
  ax.text(
      row["five_star_pct"] + 0.4,
      idx,
      (
          f"{row['five_star_pct']:.1f}%"
          f"  (n={int(row['n_reviews']):,}, k={int(row['n_products'])})"
      ),
      va="center",
      fontsize=8.5,
  )

ax.set_xlim(0, max(c_plot["five_star_pct"].max(), overall_fsr) * 1.38)
ax.set_xlabel("Five-Star Rate (%)")
ax.set_title("Five-Star Rate by Product Category (n=reviews, k=products)")
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_category_five_star.png")
plt.show()

# Figure 4: Heatmap of 5-star rating percentage and product count by Price Segment × Skin Type
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), dpi=150)

piv_fsr = (
    df.pivot_table(
        index="price_segment",
        columns="skin_type_cat",
        values="five_star",
        aggfunc="mean",
        observed=False,
    )
    * 100
)

piv_prod = df.pivot_table(
    index="price_segment",
    columns="skin_type_cat",
    values="product_id",
    aggfunc="nunique",
    observed=False,
)

sns.heatmap(
    piv_fsr,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=axes[0],
    cbar_kws={"label": "Five-Star Rate (%)"},
)
axes[0].set_title("Five-Star Rate: Price Segment x Skin Type")
axes[0].set_xlabel("Skin Type")
axes[0].set_ylabel("Price Segment")

sns.heatmap(
    piv_prod,
    annot=True,
    fmt="d",
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Unique Products"},
)
axes[1].set_title("Unique Products per Cell: Price Segment x Skin Type")
axes[1].set_xlabel("Skin Type")
axes[1].set_ylabel("Price Segment")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_price_skin_heatmap.png")
plt.show()

## Step 5. Summary of key EDA findings & AI use statement

Key Findings Summary (Part C: EDA & Visualization)
Overall Rating Distribution & Left Skew: Across all 1,093,847 reviews in analysis_ready.parquet, ratings have a mean of 4.299, a median of 5.0, and a standard deviation of 1.149. Five-star reviews (five_star == 1) account for 63.86% (698,531 reviews), and four- and five-star reviews combined represent 82.08% (897,848 reviews), whereas 1–3 star reviews make up 17.92% (195,999 reviews).   

Differences Across Price Segments (price_segment): Mid ($25-49) achieves the highest five-star rate at 64.27% (+0.41 percentage points vs. the 63.86% overall baseline) across 429,186 reviews and 896 unique products. Budget (<$25) (63.88%, 268,444 reviews, 481 products) and Premium ($50-99) (63.89%, 331,301 reviews, 725 products) sit virtually on the overall baseline. Luxury ($100+) has the lowest five-star rate at 60.94% (-2.92 pp vs. overall, and 3.33 pp below Mid) while accounting for only 5.93% of reviews (64,916 reviews across 249 products).   

Differences Across Product Categories (category): Product categories exhibit the widest dispersion (10.85 pp from highest to lowest). Five-star rates are highest in Lip Care (67.55%, +3.69 pp, 61,572 reviews), Masks (66.45%, +2.59 pp, 70,509 reviews), and Cleansers (66.17%, +2.31 pp, 200,529 reviews), and lowest in Eye Care (56.70%, -7.16 pp, 74,981 reviews, 184 products), Tools, Wellness & Self-Tan (59.76%, -4.10 pp, 28,385 reviews), and Sunscreen (60.11%, -3.75 pp, 41,135 reviews).   

Differences Across Skin Types (skin_type_cat): Reviewers in the 'Not reported' group record the highest five-star rate at 66.16% (+2.30 pp across 77,627 reviews and 1,735 products). Among reported skin types, Combination represents the largest review share (51.47%, 563,045 reviews, 63.73% five-star rate), followed by Dry (64.08%, +0.22 pp, 192,284 reviews), Normal (63.67%, -0.19 pp, 136,063 reviews), and Oily (62.90%, -0.96 pp, 124,828 reviews).   

Cross-Dimension Patterns & Product-Level Coverage: Two-way heatmaps (price_segment × skin_type_cat) show consistently lower five-star rates in Luxury ($100+) cells across skin types (e.g., 59.9% for Oily in Luxury) alongside smaller unique product counts per cell than in Mid or Premium. Because product counts vary substantially across segments and categories (such as 61 unique products in Lip Care vs. hundreds of products per cell in Mid and Premium), product-level sample sizes should be kept in mind during product-grouped modeling, clustering, and inference.   

## AI Use Statement (Part C — EDA & Visualization)
An AI assistant was used to help draft, structure, and debug Python (pandas, pyarrow, matplotlib, seaborn) exploratory data analysis and chart-generation code, organize notebook section descriptions, and draft summary wording based on computed statistics. AI assistance supported code drafting and documentation; interpretation should follow the stated product-cluster and model assumptions.   

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = output_dir("C")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

overall_reviews = len(df)
overall_mean = df["rating"].mean()
overall_fsr = df["five_star"].mean() * 100


def build_eda_summary(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
  """Build a standardized EDA summary table for a given dimension."""
  summary = data.groupby(group_col, observed=False).agg(
      n_reviews=("five_star", "count"),
      n_products=("product_id", "nunique"),
      n_five_star=("five_star", "sum"),
      mean_rating=("rating", "mean"),
      median_rating=("rating", "median"),
      std_rating=("rating", "std"),
      five_star_rate_pct=("five_star", lambda s: s.mean() * 100),
  )
  summary["review_share_pct"] = summary["n_reviews"] / overall_reviews * 100
  summary["diff_mean_vs_overall"] = summary["mean_rating"] - overall_mean
  summary["diff_fsr_vs_overall_pp"] = (
      summary["five_star_rate_pct"] - overall_fsr
  )

# Maintain the original category order (if it is an ordered categorical variable) or sort by five-star rating in descending order
  if not isinstance(data[group_col].dtype, pd.CategoricalDtype):
    summary = summary.sort_values("five_star_rate_pct", ascending=False)

  return summary.round({
      "mean_rating": 3,
      "median_rating": 1,
      "std_rating": 3,
      "five_star_rate_pct": 2,
      "review_share_pct": 2,
      "diff_mean_vs_overall": 3,
      "diff_fsr_vs_overall_pp": 2,
  })


# Summary Table of Overall Star Rating Distribution (1–5 Stars)
rating_counts = df["rating"].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
eda_by_rating = pd.DataFrame({
    "rating": rating_counts.index,
    "n_reviews": rating_counts.values,
    "review_share_pct": (rating_counts.values / overall_reviews * 100).round(2),
    "cum_share_pct": (
        (rating_counts.values / overall_reviews * 100).cumsum().round(2)
    ),
    "is_five_star": rating_counts.index == 5,
})
eda_by_rating.to_csv(OUT_DIR / "rating_distribution.csv", index=False)

# Summary Table of the Three Core Dimensions: price_segment / category / skin_type_cat
eda_by_price = build_eda_summary(df, "price_segment")
eda_by_category = build_eda_summary(df, "category")
eda_by_skin_type = build_eda_summary(df, "skin_type_cat")

eda_by_price.to_csv(OUT_DIR / "by_price_segment.csv")
eda_by_category.to_csv(OUT_DIR / "by_category.csv")
eda_by_skin_type.to_csv(OUT_DIR / "by_skin_type.csv")

# Cross-dimensional summary table: price_segment x skin_type_cat (includes review count, number of unique products, and 5-star rating percentage)
eda_by_price_skin = (
    df.groupby(["price_segment", "skin_type_cat"], observed=False)
    .agg(
        n_reviews=("five_star", "count"),
        n_products=("product_id", "nunique"),
        mean_rating=("rating", "mean"),
        five_star_rate_pct=("five_star", lambda s: s.mean() * 100),
    )
    .reset_index()
)
eda_by_price_skin["diff_fsr_vs_overall_pp"] = (
    eda_by_price_skin["five_star_rate_pct"] - overall_fsr
)
eda_by_price_skin = eda_by_price_skin.round({
    "mean_rating": 3,
    "five_star_rate_pct": 2,
    "diff_fsr_vs_overall_pp": 2,
})
eda_by_price_skin.to_csv(OUT_DIR / "by_price_skin_type.csv", index=False)

print("Exported summary tables to:", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*.csv")):
  print(" -", p.name)
